# Training a Small Language Model (TinyGPT) on a Technical QA Dataset

This notebook performs a **hyperparameter search** for a lightweight Transformer model (`TinyGPT`) and evaluates it on a QA dataset generated from a product manual.

## 🎯 Goal

Find the best **trade-off** between:
- **Model size** — footprint on embedded devices (e.g. Arduino);
- **Performance** — loss, perplexity, token accuracy, BLEU, exact match.

## 📋 Notebook Structure

| Step | Description |
|------|-------------|
| **1** | Imports and environment setup |
| **2** | Dataset loading and preprocessing |
| **3** | Hyperparameter Search (sub-sections 3.1 – 3.6) |
| **4** | Result persistence (JSON + CSV summaries) |

> 💡 The **Hyperparameter Search** section has been intentionally **split into small cells**
> so that each step of the pipeline can be inspected, modified or re-run in isolation.

> ℹ️ All code is preserved as in the original; only comments, printed messages and
> markdown explanations have been reorganised for clarity.

## 1. Imports and Environment Setup

We start by installing the project dependencies and importing the modules we need.
The two main custom modules are:
- **`src.slm.TinyGPT`** — the Transformer model itself (train/eval/save/codegen API).
- **`src.utils.dataset.QADataset`** — a `torch.utils.data.Dataset` wrapper for QA pairs.
- **`src.utils.tokenizer.CustomTokenizer`** — a lightweight word-level tokenizer.

In [1]:
# !pip install -r requirements.txt

In [2]:
# --- Optimization / loss ---
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

# --- Standard library ---
import os
import re
import unicodedata
from collections import Counter
import random

# --- Data handling ---
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# --- Custom project modules ---
from src.slm import TinyGPT
from src.utils.dataset import QADataset
from src.utils.tokenizer import CustomTokenizer

## 2. Dataset Loading and Preprocessing

We load the QA dataset from a JSONL file (`dataset_CAT-4083-UK.json`).
If the file is missing, a **synthetic fallback** dataset is generated so that the
rest of the notebook can still be executed.

Each QA pair goes through `preprocess_text`, which:
1. Lower-cases the text;
2. Removes accents (NFKD + skip combining marks);
3. Strips characters outside `[a-z0-9 .,!?'\-/:]`;
4. Collapses multiple whitespaces into one.

Then we keep only the **top `VOCAB_SIZE` words** (frequency-based) so that both
questions and answers stay inside a compact vocabulary.

In [3]:
random.seed(0)
DATASET_PATH = "./data/dataset_CAT-4083-UK.json"
VOCAB_SIZE   = 1000


def preprocess_text(text):
    """Basic text normalisation used for both questions and answers."""
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z0-9\s.,!?'\-/:]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


if not os.path.exists(DATASET_PATH):
    # --- Fallback: synthesise a very small QA corpus ---
    print("Gerando dataset sintético...")
    topics = ["temperature", "pressure", "volume", "mass", "energy",
              "velocity", "acceleration", "force"]
    generic_qna = []
    for _ in range(1000):
        t = random.choice(topics)
        q = f"What is the unit of {t}"
        a = f"The unit of {t} is " + random.choice(
            ["kg", "m/s", "J", "N", "Pa", "K", "m3", "W"])
        generic_qna.append({"question": preprocess_text(q),
                            "answer":   preprocess_text(a)})
else:
    # --- Real dataset: load JSONL + filter to top-VOCAB_SIZE words ---
    df = pd.read_json(DATASET_PATH, lines=True)
    print(f"Total de registros carregados: {len(df)}")

    # Collect all words from questions and answers
    texts = []
    for _, row in df.iterrows():
        q = preprocess_text(row["question"])
        a = preprocess_text(row["answer"])
        if q:
            texts.append(q)
        if a:
            texts.append(a)

    counter = Counter()
    for t in texts:
        counter.update(t.split())

    # Build vocabulary as the most common words
    vocab = [w for w, _ in counter.most_common(VOCAB_SIZE)]
    vocab_set = set(vocab)

    # Rebuild QA pairs keeping only in-vocabulary words
    generic_qna = []
    for _, row in df.iterrows():
        q = [w for w in preprocess_text(row["question"]).split() if w in vocab_set]
        a = [w for w in preprocess_text(row["answer"]).split() if w in vocab_set]
        if q and a:
            generic_qna.append({"question": " ".join(q),
                                "answer":   " ".join(a)})

print(f"Total de pares Q&A: {len(generic_qna)}")
for item in generic_qna[:3]:
    print("Q:", item["question"])
    print("A:", item["answer"])
    print()

Total de registros carregados: 132
Total de pares Q&A: 132
Q: what standards do the instruments meet according to the eu?
A: din/en 61000-6-2 and din/en 61000-6-3.

Q: how many pages are dedicated to pressure sensor selection guides?
A: two pages 6 and 7 .

Q: what pressure ranges do scp09 and scp10 sensors measure?
A: scp09 and scp10 measure mid and high pressure ranges.



### 2.1 Length Filtering

We discard any QA pair whose question or answer exceeds `MAX_WORDS` tokens.
This keeps `MAX_LEN` (the model context length) small and the training fast.

In [4]:
MAX_WORDS = 30

filtered_qna = [
    item for item in generic_qna
    if len(item["question"].split()) <= MAX_WORDS
    and len(item["answer"].split()) <= MAX_WORDS
]

print(f"Dataset original: {len(generic_qna)}")
print(f"Dataset filtrado: {len(filtered_qna)}")
print(f"Removidos: {len(generic_qna) - len(filtered_qna)}")

Dataset original: 132
Dataset filtrado: 132
Removidos: 0


## 3. Hyperparameter Search

### 🧠 What is hyperparameter search?

Hyperparameter search (a.k.a. *hyperparameter tuning*) consists of training
**several models** with different architectural / optimization settings and
comparing them to find the best trade-off.

In this notebook we use a simple **grid search**:

1. Define a list of candidate values for each hyperparameter;
2. Generate all combinations (Cartesian product);
3. Train one model per combination;
4. Evaluate each model and save its metrics + footprint.

### 🧩 Hyperparameters tuned

| Name | Role | Candidate values |
|------|------|------------------|
| `embedding_dim` | Token embedding dimension | `[12]` |
| `heads` | Number of attention heads | `[2]` |
| `layers` | Number of Transformer blocks | `[1]` |
| `drop_rate` | Dropout probability | `[0.01]` |
| `num_epochs` | Training epochs | `[300]` |
| `batch_size` | Batch size | `[4]` |
| `lr` | Learning rate | `[0.01]` |

> ⚠️ The grid currently has a **single combination** to keep runtime short.
> To actually perform a search, just add more values, e.g.:
> `"embedding_dim": [12, 24, 48], "layers": [1, 2, 3]`.

### 🗂️ Section subdivision

| Sub-step | Responsibility |
|----------|----------------|
| **3.1** | Experiment configuration (dataset split + tokenizer + loaders) |
| **3.2** | Utility functions (`flatten_json_to_list`, `count_total_parameters`, `get_total_cpp_header_size`) |
| **3.3** | Hyperparameter grid + Cartesian combinations |
| **3.4** | Main training + evaluation loop |
| **3.5** | Persist the summary JSON |
| **3.6** | Persist the summary CSV |

### 3.1 Experiment Configuration

Before running the search, we fix the **dataset split** and build the
**tokenizer** and **DataLoaders** that will be reused by every configuration.

Key decisions:
- The tokenizer is built from the **whole** `filtered_qna` (train + eval) so that
  every configuration sees the same vocabulary.
- `MAX_LEN = 50` is the padded sequence length.
- `TinyGPT.train()` expects a **list of QA dicts**, not a `QADataset`, so we keep
  both `train_qna` (list) and `train_dataset` (Dataset) handy.

In [5]:
# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

# Number of QA pairs used in the experiment.
n = 20  # use min(5000, len(filtered_qna)) for a larger run

train_qna_full = filtered_qna[:n]

# Train / Eval split (70 / 30)
split = int(len(train_qna_full) * 0.7)
train_qna = train_qna_full[:split]
eval_qna = train_qna_full[split:]

# --- Print dataset summary ---
print("=" * 60)
print("DATASET")
print("=" * 60)
print("Total:", len(train_qna_full))
print("Train:", len(train_qna))
print("Eval :", len(eval_qna))
print("=" * 60)

# --- Tokenizer built from the full filtered_qna ---
tokenizer = CustomTokenizer(data_source=filtered_qna)
MAX_LEN = 50
print("vocab_size:", tokenizer.vocab_size)
print("MAX_LEN:", MAX_LEN)

# --- QADataset wrappers (kept for inspection / external use) ---
train_dataset = QADataset(qna_list=train_qna, tokenizer=tokenizer, max_length=MAX_LEN)
eval_dataset = QADataset(qna_list=eval_qna, tokenizer=tokenizer, max_length=MAX_LEN)

loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=8, shuffle=False)

# --- Sanity check on one batch ---
if len(train_dataset) > 0:
    xb, yb = next(iter(loader))
    print("vocab_size:", tokenizer.vocab_size, "| xb:", xb.shape, "| yb:", yb.shape)

# Placeholder for the evaluation topics (unused downstream)
topics = [""] * len(eval_qna)

DATASET
Total: 20
Train: 14
Eval : 6
[INFO] Vocabulary built from Q&A with 489 tokens.
vocab_size: 489
MAX_LEN: 50
[INFO] Max original sequence length found: 22
[INFO] Sequences adjusted to max_length: 50
[INFO] Max original sequence length found: 20
[INFO] Sequences adjusted to max_length: 50
vocab_size: 489 | xb: torch.Size([8, 50]) | yb: torch.Size([8, 50])


### 3.2 Utility Functions

Three helpers are used to describe each trained model:

1. **`flatten_json_to_list`** — recursively collects all numeric values from a JSON file.
   Used by the parameter counter.
2. **`count_total_parameters`** — sums numeric values across `embedding.json`,
   `final_norm.json`, `out_head.json` and `transformer_blocks.json`.
   Gives us the **model footprint in parameters**.
3. **`get_total_cpp_header_size`** — sums the size (bytes) of every `.cpp` and `.h`
   file produced by the code generator. Gives us the **flash footprint** on an MCU.

In [6]:
import json
from typing import List


def flatten_json_to_list(input_json_path: str) -> List[float]:
    """
    Recursively extract every numeric value from a JSON file.
    """
    with open(input_json_path, "r") as f:
        json_data = json.load(f)

    values = []

    def extract_values(data):
        if isinstance(data, dict):
            for _, value in data.items():
                extract_values(value)
        elif isinstance(data, list):
            for item in data:
                if isinstance(item, list):
                    for num in item:
                        if isinstance(num, (int, float)):
                            values.append(num)
                elif isinstance(item, (int, float)):
                    values.append(item)
        elif isinstance(data, (int, float)):
            values.append(data)
        elif data is None:
            pass
        else:
            raise ValueError(f"Unsupported type: {type(data)}")

    extract_values(json_data)
    return values

In [7]:
def count_total_parameters(path_input: str) -> int:
    """
    Sum the number of numeric values across:
    embedding.json, final_norm.json, out_head.json, transformer_blocks.json.
    """
    file_names = [
        "embedding.json",
        "final_norm.json",
        "out_head.json",
        "transformer_blocks.json",
    ]

    total_params = 0
    for file_name in file_names:
        file_path = os.path.join(path_input, file_name)
        try:
            flattened = flatten_json_to_list(input_json_path=file_path)
            total_params += len(flattened)
        except FileNotFoundError:
            print(f"Warning: file not found: {file_name}")
        except Exception as e:
            print(f"Error processing {file_name}: {e}")

    return total_params

In [8]:
def get_total_cpp_header_size(folder_path: str) -> int:
    """
    Sum the size (bytes) of every .cpp / .h file in the given folder.
    """
    total_size = 0
    if not os.path.exists(folder_path):
        print(f"Warning: directory not found: {folder_path}")
        return 0

    for file_name in os.listdir(folder_path):
        if file_name.endswith((".cpp", ".h")):
            file_path = os.path.join(folder_path, file_name)
            total_size += os.path.getsize(file_path)

    return total_size

### 3.3 Hyperparameter Grid

The grid is a dictionary mapping each hyperparameter to a **list of values**.
We then compute the **Cartesian product** via `itertools.product`, which yields
one tuple per combination.

Each combination is later converted into a `params` dict and given a
human-readable `config_id`, e.g.:

```
emb12_h2_l1_d001_ep300_bs4_lr001
```

That ID is used to name every artefact produced by the experiment
(saved model, Arduino code, evaluation JSONs).

`all_results` is the accumulator that will hold one entry per combination.

In [9]:
import itertools
from typing import Any, Dict

# ============================================================
# HYPERPARAMETER GRID
# ============================================================
param_grid = {
    "embedding_dim": [12],
    "heads":         [2],
    "layers":        [1],
    "drop_rate":     [0.01],
    "num_epochs":    [300],
    "batch_size":    [4],
    "lr":            [0.01],
}

# ============================================================
# GENERATE COMBINATIONS
# ============================================================
keys = param_grid.keys()
values = param_grid.values()
combinations = list(itertools.product(*values))

print()
print("=" * 60)
print("HYPERPARAMETER SEARCH")
print("=" * 60)
print("Total configurations:", len(combinations))
print("=" * 60)

# Accumulator for all experiments
all_results: List[Dict[str, Any]] = []


HYPERPARAMETER SEARCH
Total configurations: 1


### 3.4 Main Training + Evaluation Loop

For **each combination** in the grid we:

1. Build a unique `config_id` string.
2. Create dedicated output folders (`output_dir_*` and `output_arduino_*`).
3. Instantiate a fresh `TinyGPT`.
4. Call `model.train(...)` on the **list of QA dicts** `train_qna`.
5. Save the model (`model.save`) and generate Arduino-compatible C++ code
   (`model.code_generator`).
6. Evaluate on **train** and **eval** sets (`model.evaluation`) and persist both
   metric JSONs.
7. Count parameters and C++ footprint.
8. Append a structured `result_entry` to `all_results`.

> ⚠️ **Note on `train_qna` vs `train_dataset`:**
> `TinyGPT.train()` expects a **list of dicts** (`{"question": ..., "answer": ...}`),
> not a `QADataset`. Passing the Dataset will silently fail or misbehave.

In [10]:
# ============================================================
# MAIN LOOP
# ============================================================
for idx, combo in enumerate(combinations):

    # --------------------------------------------------------
    # 1. Build the configuration dict and its ID
    # --------------------------------------------------------
    params = dict(zip(keys, combo))

    config_id = (
        f"emb{params['embedding_dim']}_"
        f"h{params['heads']}_"
        f"l{params['layers']}_"
        f"d{str(params['drop_rate']).replace('.', '')}_"
        f"ep{params['num_epochs']}_"
        f"bs{params['batch_size']}_"
        f"lr{str(params['lr']).replace('.', '')}"
    )

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------
    print()
    print("=" * 60)
    print(f"Running configuration {idx + 1}/{len(combinations)}")
    print(f"Configuration ID: {config_id}")
    print(f"Parameters: {params}")
    print("=" * 60)

    # --------------------------------------------------------
    # 2. Output directories (one per configuration)
    # --------------------------------------------------------
    output_model_dir   = f"output_dir_{config_id}"
    output_arduino_dir = f"output_arduino_{config_id}"
    os.makedirs(output_model_dir,   exist_ok=True)
    os.makedirs(output_arduino_dir, exist_ok=True)

    # --------------------------------------------------------
    # 3. Instantiate a fresh model
    # --------------------------------------------------------
    model = TinyGPT(
        embedding_dim=params["embedding_dim"],
        heads=params["heads"],
        layers=params["layers"],
        drop_rate=params["drop_rate"],
    )

    # --------------------------------------------------------
    # 4. Train on the LIST of QA dicts
    # --------------------------------------------------------
    print()
    print("Training...")
    model.train(
        synthetic_dataset=train_qna,
        num_epochs=params["num_epochs"],
        batch_size=params["batch_size"],
        lr=params["lr"],
        optimizer_cls=AdamW,
        loss_fn_cls=CrossEntropyLoss,
    )

    # --------------------------------------------------------
    # 5. Persist model + generate C++ code
    # --------------------------------------------------------
    model.save(output_folder=output_model_dir)
    model.code_generator(
        model_folder=output_model_dir,
        output_arduino_folder=output_arduino_dir,
        model_name=f"model_{config_id}",
    )

    # --------------------------------------------------------
    # 6. Evaluation on TRAIN and EVAL sets
    # --------------------------------------------------------
    print()
    print("Evaluating train set...")
    train_metrics = model.evaluation(
        train_qna,
        batch_size=32,
        num_examples=len(train_qna),
    )

    print("Evaluating eval set...")
    eval_metrics = model.evaluation(
        eval_qna,
        batch_size=32,
        num_examples=len(eval_qna),
    )

    # Persist both metric dictionaries as JSON
    train_json_path = f"evaluation_train_{config_id}.json"
    with open(train_json_path, "w", encoding="utf-8") as f:
        json.dump(train_metrics, f, indent=2, ensure_ascii=False)

    eval_json_path = f"evaluation_eval_{config_id}.json"
    with open(eval_json_path, "w", encoding="utf-8") as f:
        json.dump(eval_metrics, f, indent=2, ensure_ascii=False)

    # --------------------------------------------------------
    # 7. Footprint metrics
    # --------------------------------------------------------
    num_params = count_total_parameters(output_model_dir)
    cpp_size   = get_total_cpp_header_size(output_arduino_dir)

    # --------------------------------------------------------
    # 8. Store the result entry
    # --------------------------------------------------------
    result_entry = {
        "config_id":        config_id,
        "params":           params,
        "train_metrics":    train_metrics,
        "eval_metrics":     eval_metrics,
        "train_json":       train_json_path,
        "eval_json":        eval_json_path,
        "model_dir":        output_model_dir,
        "arduino_dir":      output_arduino_dir,
        "number_parameter": num_params,
        "cpp_model_size":   cpp_size,
    }
    all_results.append(result_entry)

    # --------------------------------------------------------
    # Console summary for this configuration
    # --------------------------------------------------------
    train_loss = train_metrics.get("avg_loss",       float("nan"))
    train_acc  = train_metrics.get("token_accuracy", float("nan"))
    eval_loss  = eval_metrics.get("avg_loss",        float("nan"))
    eval_acc   = eval_metrics.get("token_accuracy",  float("nan"))

    print()
    print("-" * 60)
    print(f"Train - Avg loss: {train_loss:.4f}")
    print(f"Train - Token accuracy: {train_acc:.4f}")
    print(f"Eval  - Avg loss: {eval_loss:.4f}")
    print(f"Eval  - Token accuracy: {eval_acc:.4f}")
    print(f"Model parameters: {num_params}")
    print(f"C++ code size: {cpp_size} bytes")
    print("-" * 60)


Running configuration 1/1
Configuration ID: emb12_h2_l1_d001_ep300_bs4_lr001
Parameters: {'embedding_dim': 12, 'heads': 2, 'layers': 1, 'drop_rate': 0.01, 'num_epochs': 300, 'batch_size': 4, 'lr': 0.01}

Training...
[INFO] Vocabulary built from Q&A with 112 tokens.
[INFO] Context length: 24
[INFO] Max original sequence length found: 22
[INFO] Sequences adjusted to max_length: 24
[INFO] Using Apple MPS (Metal Performance Shaders).
                  TRAINING MODEL                  
[INFO] Epoch 1/300 - Loss: 4.8435
[INFO] Epoch 2/300 - Loss: 4.4778
[INFO] Epoch 3/300 - Loss: 4.2094
[INFO] Epoch 4/300 - Loss: 3.9947
[INFO] Epoch 5/300 - Loss: 3.7523
[INFO] Epoch 6/300 - Loss: 3.5386
[INFO] Epoch 7/300 - Loss: 3.3020
[INFO] Epoch 8/300 - Loss: 3.0814
[INFO] Epoch 9/300 - Loss: 2.8725
[INFO] Epoch 10/300 - Loss: 2.7025
[INFO] Epoch 11/300 - Loss: 2.5014
[INFO] Epoch 12/300 - Loss: 2.2747
[INFO] Epoch 13/300 - Loss: 2.0995
[INFO] Epoch 14/300 - Loss: 1.9305
[INFO] Epoch 15/300 - Loss: 1.779

### 3.5 Persist Summary JSON

We dump `all_results` — one entry per trained configuration, containing both
the hyperparameters and the collected metrics — into
`hyperparameter_search_summary.json`.

This file is the source of truth for any downstream analysis.

In [11]:
summary_path = "hyperparameter_search_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print()
print("=" * 60)
print("Hyperparameter search completed!")
print(f"Results saved to '{summary_path}'.")
print("=" * 60)


Hyperparameter search completed!
Results saved to 'hyperparameter_search_summary.json'.


### 3.6 Persist Summary CSV

Finally, we flatten each `result_entry` into a **flat row** and write a CSV.
This is convenient for:
- Quick inspection in Excel / LibreOffice / pandas;
- Building Pareto-frontier and parallel-coordinate plots in the follow-up notebook.

The `fieldnames` list defines both the **column order** and which metrics are exported.

In [12]:
import csv

csv_path = "hyperparameter_search_summary.csv"

fieldnames = [
    "config_id",
    "embedding_dim",
    "heads",
    "layers",
    "drop_rate",
    "num_epochs",
    "batch_size",
    "lr",

    # Train metrics
    "train_avg_loss",
    "train_perplexity",
    "train_token_accuracy",
    "train_num_samples",
    "train_avg_bleu",
    "train_exact_match_rate",

    # Eval metrics
    "eval_avg_loss",
    "eval_perplexity",
    "eval_token_accuracy",
    "eval_num_samples",
    "eval_avg_bleu",
    "eval_exact_match_rate",

    # Model footprint
    "number_parameter",
    "cpp_model_size",
]

with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for entry in all_results:
        train      = entry["train_metrics"]
        evaluation = entry["eval_metrics"]

        row = {
            # Configuration
            "config_id":     entry["config_id"],
            "embedding_dim": entry["params"]["embedding_dim"],
            "heads":         entry["params"]["heads"],
            "layers":        entry["params"]["layers"],
            "drop_rate":     entry["params"]["drop_rate"],
            "num_epochs":    entry["params"]["num_epochs"],
            "batch_size":    entry["params"]["batch_size"],
            "lr":            entry["params"]["lr"],

            # Train
            "train_avg_loss":        train.get("avg_loss",          ""),
            "train_perplexity":      train.get("perplexity",        ""),
            "train_token_accuracy":  train.get("token_accuracy",    ""),
            "train_num_samples":     train.get("num_samples",       ""),
            "train_avg_bleu":        train.get("avg_bleu",          ""),
            "train_exact_match_rate":train.get("exact_match_rate",  ""),

            # Eval
            "eval_avg_loss":         evaluation.get("avg_loss",         ""),
            "eval_perplexity":       evaluation.get("perplexity",       ""),
            "eval_token_accuracy":   evaluation.get("token_accuracy",   ""),
            "eval_num_samples":      evaluation.get("num_samples",      ""),
            "eval_avg_bleu":         evaluation.get("avg_bleu",         ""),
            "eval_exact_match_rate": evaluation.get("exact_match_rate", ""),

            # Model
            "number_parameter": entry["number_parameter"],
            "cpp_model_size":   entry["cpp_model_size"],
        }
        writer.writerow(row)

print(f"Tabular summary saved to '{csv_path}'.")

Tabular summary saved to 'hyperparameter_search_summary.csv'.


## 4. Next Steps

The two summary files produced above are the inputs for the **analysis notebook**:

- `hyperparameter_search_summary.json` — full nested metrics per configuration;
- `hyperparameter_search_summary.csv`  — flat, one-row-per-configuration table.

Typical follow-ups:

1. **Pareto frontier** — plot `number_parameter` vs `eval_avg_loss` and mark the
   non-dominated points.
2. **Parallel coordinates** — visualise how each hyperparameter correlates with
   the metrics across the grid.
3. **Best model selection** — pick the configuration with the lowest `eval_avg_loss`
   for a given parameter budget (e.g. `< 10k params`, `< 100 kB flash`).
4. **Deployment** — compile the generated Arduino `.h` / `.cpp` for the target MCU
   and validate latency / memory in situ.